# Transfer Learning con texto: entrenar solo la cabeza vs. ajustar toda la red

## ¿Qué vamos a hacer en este ejercicio?

Vamos a construir un clasificador sencillo que lea una frase en inglés y estime si expresa un sentimiento **positivo** o **negativo**. Por ejemplo:

- `I love this movie` → positivo.
- `This film is terrible` → negativo.

Para hacerlo usaremos **DistilBERT**, un modelo de lenguaje que ya fue preentrenado con grandes cantidades de texto. No lo vamos a enseñar a comprender inglés desde cero: aprovecharemos lo que ya aprendió y veremos dos maneras de adaptarlo a nuestra tarea.

### Los dos experimentos que vamos a comparar

1. **Solo entrenar la cabeza:** dejamos congelado DistilBERT. Sus parámetros no cambian; entrenamos únicamente una pequeña capa nueva que decide entre positivo y negativo.
2. **Fine-tuning:** comenzamos también con DistilBERT preentrenado, pero permitimos que sus parámetros se ajusten a las frases de nuestro problema. La cabeza nueva también se entrena.

### ¿Qué significa “cabeza”?

La *cabeza* es la parte final que añadimos al modelo para convertir la representación de una frase en una decisión concreta. En este caso, transforma la información extraída por DistilBERT en una puntuación de sentimiento positivo o negativo.

### ¿Qué aprenderás al terminar?

- Cómo se convierte el texto en números que una red neuronal puede procesar.
- Qué partes de un modelo preentrenado se congelan o se actualizan.
- Cómo se entrena y evalúa cada alternativa.
- Cómo comparar las pérdidas de entrenamiento y las predicciones.

**Importante:** usaremos solo 24 frases de ejemplo para que el ejercicio sea pequeño y fácil de ejecutar. Es una demostración didáctica, no un experimento suficiente para decidir qué método funciona mejor en aplicaciones reales. Las frases están en inglés porque DistilBERT se preentrenó con texto en inglés. La primera ejecución requiere conexión a Internet para descargar el modelo.

## Mapa del ejercicio

| Etapa | Qué haremos | Para qué sirve |
|---|---|---|
| Preparación | Instalar e importar herramientas | Dejar listo el entorno de Colab |
| Datos | Etiquetar frases positivas y negativas | Definir qué debe aprender el clasificador |
| Tokenización | Convertir texto en IDs numéricos | Preparar las entradas para DistilBERT |
| Modelo | Conectar DistilBERT con una cabeza nueva | Obtener una predicción de sentimiento |
| Entrenamiento A | Congelar DistilBERT | Aprender solo la cabeza |
| Entrenamiento B | Descongelar DistilBERT | Ajustar el encoder y la cabeza |
| Evaluación | Probar con frases reservadas | Comparar predicciones y pérdidas |

Primero ejecutaremos el método A y después el B; al final compararemos los resultados.


## Paso 1. Instalar las bibliotecas necesarias

**¿Qué hacemos?** Instalamos `transformers`, una biblioteca que facilita descargar y utilizar modelos de lenguaje preentrenados como DistilBERT.

**¿Por qué hace falta?** PyTorch permite construir y entrenar redes neuronales; `transformers` nos proporciona el modelo preentrenado y su tokenizador.

**Qué debes hacer:** ejecuta esta celda y espera a que termine. Si Colab muestra que la instalación terminó correctamente, continúa con el paso 2. No necesitas instalar Python en tu computador: Colab ejecuta el código en su propio entorno.

In [ ]:
!pip -q install transformers

## Paso 2. Importar herramientas y preparar la ejecución

**¿Qué hacemos?** Importamos las bibliotecas que usaremos para manejar datos, cargar DistilBERT, construir el clasificador, entrenarlo y dibujar resultados.

También fijamos una **semilla aleatoria**. Esto ayuda a que algunas decisiones aleatorias sean más reproducibles, aunque no garantiza resultados idénticos en todos los equipos.

Finalmente, el código selecciona el dispositivo de cálculo:
- `cuda`: si Colab ofrece una GPU compatible.
- `cpu`: si no hay GPU disponible.

**Qué debes observar:** la salida indica el dispositivo elegido. Ambos funcionan; con GPU el entrenamiento normalmente es más rápido.

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModel

SEMILLA = 42
random.seed(SEMILLA)
np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEMILLA)

dispositivo = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Dispositivo:', dispositivo)

## Paso 3. Crear los datos de ejemplo

**¿Qué hacemos?** Creamos 24 frases en inglés y asignamos a cada una una etiqueta conocida:

- `1` = sentimiento positivo.
- `0` = sentimiento negativo.

Estas etiquetas son las respuestas correctas con las que el modelo aprenderá.

**Separar entrenamiento y prueba:** usamos las primeras 16 frases para entrenar y dejamos las últimas 8 para probar el modelo con frases que no utilizó durante el entrenamiento. Ambos experimentos usarán exactamente la misma separación para que la comparación sea más justa.

**Limitación importante:** el conjunto es muy pequeño y algunas expresiones son sencillas. Por eso, las métricas de prueba pueden cambiar mucho con solo una predicción diferente. El propósito principal es entender el procedimiento.

In [ ]:
textos = [
    'I love this movie',
    'This film is wonderful',
    'The food was delicious',
    'I am very happy',
    'This is an excellent product',
    'The experience was amazing',
    'I really enjoyed it',
    'Everything was fantastic',
    'I hate this movie',
    'This film is terrible',
    'The food was disgusting',
    'I am very disappointed',
    'This product is awful',
    'The experience was horrible',
    'I really disliked it',
    'Everything was bad',
    'The service was excellent',
    'What a wonderful day',
    'The product broke immediately',
    'I regret buying this',
    'I am delighted with the result',
    'This was a complete disaster',
    'The quality is fantastic',
    'I would never recommend it'
]

# 1 = positivo; 0 = negativo
etiquetas = np.array([
    1, 1, 1, 1, 1, 1, 1, 1,
    0, 0, 0, 0, 0, 0, 0, 0,
    1, 1, 0, 0, 1, 0, 1, 0
], dtype=np.float32)

# Partición fija para que ambos experimentos utilicen los mismos datos
train_texts = textos[:16]
train_labels = etiquetas[:16]
test_texts = textos[16:]
test_labels = etiquetas[16:]

print('Frases totales:', len(textos))
print('Entrenamiento:', len(train_texts))
print('Prueba:', len(test_texts))
print('Etiquetas de entrenamiento:', train_labels.astype(int))

## Paso 4. Convertir las frases en números y cargar DistilBERT

Las redes neuronales no reciben palabras directamente: necesitan números. El **tokenizador** divide cada frase en unidades de texto (*tokens*) y convierte esas unidades en identificadores numéricos.

Por ejemplo, una frase como `I love this movie` se transforma en una secuencia de IDs. El tokenizador también:
- agrega relleno (`padding`) para que las frases de un mismo lote tengan la misma longitud;
- limita la longitud máxima (`max_length=64`);
- genera una máscara de atención para distinguir los tokens reales del relleno.

Después descargamos DistilBERT, que ya fue preentrenado. En este ejercicio reutilizamos ese conocimiento previo para analizar sentimientos, en vez de aprender todas las representaciones del lenguaje desde cero.

**Qué debes observar:** las dimensiones impresas indican cuántas frases y posiciones numéricas hay en los datos. La lista de tokens y de IDs muestra cómo una frase se convierte en números. La primera descarga necesita Internet.

In [ ]:
modelo_nombre = 'distilbert/distilbert-base-uncased'
tokenizador = AutoTokenizer.from_pretrained(modelo_nombre)

X_train = tokenizador(
    train_texts, padding=True, truncation=True,
    max_length=64, return_tensors='pt'
)
X_test = tokenizador(
    test_texts, padding=True, truncation=True,
    max_length=64, return_tensors='pt'
)

print('Forma de input_ids para entrenamiento:', tuple(X_train['input_ids'].shape))
print('Forma de input_ids para prueba:', tuple(X_test['input_ids'].shape))
print('\nEjemplo de frase:', train_texts[0])
print('Tokens:', tokenizador.tokenize(train_texts[0]))
print('IDs numéricos:', X_train['input_ids'][0].tolist())

## Paso 5. Construir el modelo y añadir la cabeza de clasificación

Aquí conectamos dos partes:

1. **Encoder (DistilBERT):** recibe los IDs y produce representaciones numéricas contextualizadas de los tokens. En términos sencillos, transforma la frase en una representación que recoge información sobre su contenido.
2. **Cabeza de clasificación:** una capa lineal pequeña que usa esa representación para producir una puntuación de sentimiento.

El código combina las representaciones de los tokens reales para obtener un vector por frase; la máscara evita incluir el relleno en el promedio. La cabeza produce un *logit* (una puntuación que luego se puede convertir en probabilidad).

El argumento `congelar` define el experimento:
- `congelar=True`: los parámetros de DistilBERT no se actualizan. Aprende solo la cabeza nueva.
- `congelar=False`: se actualizan tanto DistilBERT como la cabeza.

**Idea clave:** en ambos métodos partimos de DistilBERT preentrenado. La diferencia es qué parámetros dejamos aprender durante esta tarea.

In [ ]:
class ClasificadorSentimiento(nn.Module):
    def __init__(self, congelar=True):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(modelo_nombre)
        self.congelar = congelar
        self.cabeza = nn.Linear(self.encoder.config.hidden_size, 1)

        for parametro in self.encoder.parameters():
            parametro.requires_grad = not congelar

    def forward(self, input_ids, attention_mask):
        if self.congelar:
            with torch.no_grad():
                salida = self.encoder(
                    input_ids=input_ids,
                    attention_mask=attention_mask
                )
        else:
            salida = self.encoder(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

        vectores = salida.last_hidden_state
        mascara = attention_mask.unsqueeze(-1).float()
        vector_frase = (vectores * mascara).sum(dim=1) / mascara.sum(dim=1).clamp(min=1)
        logits = self.cabeza(vector_frase).squeeze(-1)
        return logits

def contar_parametros(modelo):
    total = sum(p.numel() for p in modelo.parameters())
    entrenables = sum(p.numel() for p in modelo.parameters() if p.requires_grad)
    print(f'Parámetros totales: {total:,}')
    print(f'Parámetros entrenables: {entrenables:,}')
    print(f'Porcentaje entrenable: {100*entrenables/total:.3f}%')

## Paso 6. Preparar la función de entrenamiento

Esta función contiene el procedimiento que se repetirá en los dos experimentos.

En cada época, el modelo:
1. recibe un pequeño lote de frases y sus etiquetas;
2. produce puntuaciones;
3. compara las puntuaciones con las etiquetas correctas mediante una función de pérdida;
4. calcula cómo deben cambiar los parámetros;
5. actualiza los parámetros que estén habilitados para aprender.

La **pérdida** (*loss*) mide qué tan lejos están las predicciones de las respuestas correctas; durante el entrenamiento esperamos que, en general, disminuya.

Usamos tasas de aprendizaje diferentes:
- En el método “solo la cabeza”, se actualiza únicamente la cabeza.
- En fine-tuning, se usa una tasa pequeña para DistilBERT, porque ya tiene pesos útiles, y una tasa mayor para la cabeza nueva.

Para que el ejemplo sea breve usamos pocas épocas. No significa que tres épocas sean óptimas para cualquier problema.

In [ ]:
def entrenar_modelo(congelar, epocas=3, batch_size=4):
    modelo = ClasificadorSentimiento(congelar=congelar).to(dispositivo)
    contar_parametros(modelo)

    if congelar:
        grupos = [{'params': modelo.cabeza.parameters(), 'lr': 1e-3}]
    else:
        grupos = [
            {'params': modelo.encoder.parameters(), 'lr': 2e-5},
            {'params': modelo.cabeza.parameters(), 'lr': 1e-3}
        ]

    optimizador = torch.optim.AdamW(grupos)
    funcion_perdida = nn.BCEWithLogitsLoss()

    conjunto = TensorDataset(
        X_train['input_ids'], X_train['attention_mask'], torch.tensor(train_labels)
    )
    generador = torch.Generator().manual_seed(SEMILLA)
    cargador = DataLoader(conjunto, batch_size=batch_size, shuffle=True, generator=generador)
    historial = []

    for epoca in range(epocas):
        modelo.train()
        if congelar:
            # Mantener DistilBERT en modo evaluación y congelado.
            modelo.encoder.eval()

        perdida_acumulada = 0.0
        for ids, mascara, y in cargador:
            ids = ids.to(dispositivo)
            mascara = mascara.to(dispositivo)
            y = y.to(dispositivo)

            optimizador.zero_grad()
            logits = modelo(ids, mascara)
            perdida = funcion_perdida(logits, y)
            perdida.backward()
            optimizador.step()
            perdida_acumulada += perdida.item()

        promedio = perdida_acumulada / len(cargador)
        historial.append(promedio)
        print(f'Época {epoca+1}/{epocas} | pérdida: {promedio:.4f}')

    return modelo, historial

## Paso 7. Experimento A: entrenar solo la cabeza

**Objetivo:** comprobar cómo se comporta el clasificador nuevo cuando las representaciones de DistilBERT permanecen fijas.

Durante este experimento:
- DistilBERT está congelado: no modifica sus pesos.
- La cabeza de clasificación sí aprende a distinguir frases positivas de negativas.
- Como se actualizan muchos menos parámetros, este método suele requerir menos tiempo y memoria.

**Analogía:** imaginemos que una persona experta ya sabe describir el contenido de una frase. En este paso no volvemos a enseñarle inglés; solo entrenamos a otra persona para que use esas descripciones y decida si el sentimiento es positivo o negativo.

In [ ]:
modelo_A, perdida_A = entrenar_modelo(
    congelar=True,
    epocas=3
)


## Paso 8. Experimento B: fine-tuning de DistilBERT

**Objetivo:** permitir que el modelo adapte parte de lo que ya aprendió a nuestra tarea de análisis de sentimientos.

Aquí:
- DistilBERT comienza otra vez desde sus pesos preentrenados originales.
- Se entrena la cabeza nueva.
- También se permite actualizar los parámetros del encoder con una tasa de aprendizaje pequeña.

Creamos una instancia nueva para que el experimento B no herede los cambios del experimento A. Así comparamos las dos estrategias partiendo del mismo modelo preentrenado.

**Ventaja potencial:** las representaciones pueden especializarse mejor en el sentimiento. **Riesgo:** con tan pocos ejemplos, el modelo puede memorizar las frases de entrenamiento y generalizar mal. Además, el entrenamiento suele ser más costoso.

In [ ]:
modelo_B, perdida_B = entrenar_modelo(
    congelar=False,
    epocas=3
)


## Paso 9. Evaluar los modelos con frases que no vieron durante el entrenamiento

Ahora probamos ambos modelos con las mismas 8 frases reservadas.

Para cada frase, el código muestra:
- la predicción del modelo: `POSITIVO` o `NEGATIVO`;
- la probabilidad estimada de la clase positiva;
- la etiqueta real que asignamos al comienzo.

La **exactitud** (*accuracy*) es la proporción de frases clasificadas correctamente. Por ejemplo, acertar 6 de 8 equivale a una exactitud del 75 %.

**Cómo interpretar el resultado:** no debemos asumir que el método con mayor exactitud aquí será siempre mejor. Con solo 8 frases de prueba, cada acierto cambia la exactitud en 12,5 puntos porcentuales. El resultado sirve para practicar la evaluación, no para establecer una conclusión definitiva.

In [ ]:
def evaluar_modelo(modelo, nombre):
    modelo.eval()
    ids = X_test['input_ids'].to(dispositivo)
    mascara = X_test['attention_mask'].to(dispositivo)
    with torch.no_grad():
        logits = modelo(ids, mascara)
        probabilidades = torch.sigmoid(logits).cpu().numpy()

    predicciones = (probabilidades >= 0.5).astype(int)
    exactitud = (predicciones == test_labels.astype(int)).mean()
    print(f'\n{nombre} | exactitud de prueba: {exactitud:.1%}')
    for frase, prob, pred, real in zip(test_texts, probabilidades, predicciones, test_labels):
        clase = 'POSITIVO' if pred == 1 else 'NEGATIVO'
        etiqueta_real = 'POSITIVO' if real == 1 else 'NEGATIVO'
        print(f'{frase:36s} -> {clase:8s} (p={prob:.3f}; real={etiqueta_real})')
    return exactitud

acc_A = evaluar_modelo(modelo_A, 'A. SOLO LA CABEZA')
acc_B = evaluar_modelo(modelo_B, 'B. FINE-TUNING')

## Paso 10. Comparar las curvas de pérdida

La gráfica muestra cómo cambia la pérdida sobre los datos de entrenamiento en cada época para los dos métodos.

**Cómo leerla:**
- Eje horizontal: época, es decir, una pasada por los datos de entrenamiento.
- Eje vertical: pérdida. En general, una pérdida menor indica que el modelo se ajusta mejor a esos datos.
- Cada línea representa una estrategia: solo la cabeza o fine-tuning.

Si una línea baja más rápido, significa que esa estrategia está reduciendo más rápidamente su pérdida de entrenamiento en este experimento. **Pero una pérdida de entrenamiento menor no garantiza que el modelo funcione mejor con frases nuevas.** Para saberlo con más confianza necesitaríamos más datos, validación y pruebas repetidas.

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(range(1, len(perdida_A)+1), perdida_A, marker='o', label='Solo la cabeza')
plt.plot(range(1, len(perdida_B)+1), perdida_B, marker='s', label='Fine-tuning')
plt.xlabel('Época')
plt.ylabel('Pérdida de entrenamiento')
plt.title('Comparación de las dos estrategias')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

print(f'Exactitud solo la cabeza: {acc_A:.1%}')
print(f'Exactitud fine-tuning:    {acc_B:.1%}')

## Conclusiones

- **Solo la cabeza:** DistilBERT extrae características fijas; solo aprende el clasificador nuevo. Normalmente es más rápido y requiere menos memoria.
- **Fine-tuning:** DistilBERT y la cabeza pueden adaptarse a la tarea. Suele requerir más tiempo y puede sobreajustarse si hay pocos ejemplos.
- En ambos casos se parte de pesos preentrenados, no de pesos aleatorios para el encoder.
- Este experimento es ilustrativo: 24 frases —y solo 8 para prueba— no bastan para concluir qué método es mejor en un problema real.

### Resumen en una frase

**Transfer learning consiste en reutilizar lo que un modelo ya aprendió y adaptarlo a una tarea nueva; aquí comparamos adaptar solo la capa final frente a adaptar también el modelo preentrenado.**

### Cómo ejecutar el cuaderno

En Google Colab, selecciona **Entorno de ejecución → Ejecutar todas**. Espera a que cada celda termine antes de interpretar los resultados. La primera ejecución descargará DistilBERT desde Internet. Si Colab pierde la conexión, vuelve a conectar el entorno y ejecuta las celdas en orden.

### Preguntas para comprobar lo aprendido

1. ¿Qué parte del modelo aprende en el método “solo la cabeza”?
2. ¿Qué cambia en el fine-tuning?
3. ¿Por qué usamos las mismas frases de prueba en ambos métodos?
4. ¿Por qué no podemos concluir cuál método es mejor con solo 24 frases?